In [8]:
import sys
import pandas as pd
import numpy as np
from pyfaidx import Fasta
import os
import glob
sys.path.append('./data/')
sys.path.append('./metagenomic/')
sys.path.append('./DNAshape/')
sys.path.append('./RNA_structure/')
sys.path.append('./expression/')
sys.path.append('./methylation_lcc/')
sys.path.append('./competition/')
sys.path.append('./data/')
from metagenomic_feats import calc_metagenomic_feats
from DNAshape_feats import calc_DNAshape, make_DNAshape_df
from RNA_structure_feats import calc_RNA_structure_feats, make_RNA_struct_df
from expression_feats import calc_CUB_feats
from methylation_lcc_feats import calc_methylation_feats,calc_lcc_feats
from competition_feats import calc_competition_feats

this code assumes each row in your df contains the following columns:

g_rna_info - string in the format of 'chromsome;start_site;stop_site;strand' or with _ for human

target_seq - 20 nt seq of site

up_seq - 20 nt seq of upstream to site

down_seq - 20 nt seq of downstream to site

***************************
it assumes you have the variable:

genome - loaded genome of your organism using pyfaix's Fasta function
***************************for CUB features and competition-based features - these files are needed in the data folder

cdss - csv containing the cds's of the organism 

orfs - fasta file containing the orfs of the organism

aa_feqs - pickle file containing the amino acid frequencies based on the entire genome of the organism

codon_feqs - pickle file containing the codon frequencies based on the entire genome of the organism

SA_cod - pickle file containing the suffix array (needed for ChimeraARS) based on the entire genome of the organism

for each of these see relevant file in data folder for example

In [2]:
# tomato_genome = Fasta('./data/tomato_data/SollycM82_v1.0.fasta')
# prawn_genome = Fasta('./data/prawn_data/GCF_040412425.1_ASM4041242v1_genomic.fna')
# fly_genome = Fasta('./data/fly_data/GCF_905115235.1_iHerIll2.2.curated.20191125_genomic.fna')
# human_genome = Fasta('./data/human_data/hg38.fa')

In [ ]:
df = pd.read_csv('T_df.csv')
df = df[['g_rna_info','target_seq','up_seq','down_seq']]

In [6]:
methylation_df = calc_methylation_feats(target_seq_df)

In [ ]:
organism_name = '' # human, prawn, fly, tomato
CUB_df = calc_CUB_feats(target_seq_df,genome,organism=organism_name,calc_CAI_chimera='all')

In [ ]:
RNA_struct_df = calc_RNA_structure_feats(target_seq_df)

In [ ]:
metagenomic_df = calc_metagenomic_feats(target_seq_df)

In [ ]:
DNAshape_df = calc_DNAshape(target_seq_df,genome)

# Competition-based features
# Please note - these calculations are very heavy computationally and take a long time to run!

To evaluate the competition features, you need, in addition to the target sites, the genome and genes of the organism. The code below expects the following objects to exist:

- "sites" DataFrame of the target site with columns sequence (the target site without PAM) and chr (its chromosome id that matches the chromosome column of the genome dataframe) and site_start (the start coordinate of the target site in the genome on the respective chromosome) and is_NGG_PAM, a boolean that indicates whether the target site has a PAM that follows the NGG pattern.
- "comp_genome" DataFrame is the host genome with the columns seq, chromosome
- "genes" DataFrame with the columns seq and full_seq, where seq is the coding sequence and full_seq is the coding sequence including introns. We filter only one (the longest) isoform for each gene to not double count sites that appear in multiple isoforms.

It will then add the competitor site counts to the sites dataframe.
Below I give an example of the input data and compute the features for a random example site.

In [ ]:
competition_df = calc_competition_feats(df,comp_genome,genes)

In [ ]:
all_feats_df = pd.concat([methylation_df, CUB_df, RNA_struct_df, metagenomic_df, DNAshape_df, competition_df],axis=1)

In [34]:
def make_rep_df(df):
    feats_to_keep = []
    cai_and_chimera_cols = [f'{name}_{f}{m}_{win}'
        for win in [0] 
        for name,f in zip(['codon','aa','CAI','chimera'],['freqs_','freqs_','','']) 
        for m in ['avg']]
    # tad = ['tad_density','tad_angle','tad_interactions']
    # feats_to_keep.extend(tad)
    # orenstein = ['RRBS','H3K4me3','Dnase','CTCF']
    # feats_to_keep.extend(orenstein)
    isana = ['g_DNADNA','g_RNADNA','guideEne','guide&scafEne','isBasePairs','Head1','Head2','Head3','1&2&3']
    feats_to_keep.extend(cai_and_chimera_cols)
    feats_to_keep.extend(isana)
    feats_to_keep.extend([x for x in df.columns.to_list() if 'DNAshape' in x 
                  and 'mean' in x and 'ext' not in x])
    feats_to_keep.extend([x for x in df.columns.to_list() if 'enthalpy' in x
                      and 'mean' in x and 'ext' not in x])
    LCC_feats = [x for x in df.columns.to_list() if 'LCC' in x]
    df.loc[:,'LCC_rep'] = df[LCC_feats].mean(axis=1)
    feats_to_keep.append('LCC_rep')
    feats_to_keep.extend([x for x in df.columns.to_list() if 'methylation' in x
                      and 'mean' in x and 'ext' not in x])
    sites_feats = [x for x in df.columns.to_list() if 'sites_count' in x
     and (x[-2:] in '-0-5' or x[-2:] in 'd0d2d5')
     and 'orf' not in x and 'full' not in x and 'global' not in x
     and(('ngg_pam_1000_' in x or 'count_1000_' in x)
     or ('ngg_pam_10000_' in x or 'count_10000_' in x)
     or ('ngg_pam_100000_' in x or 'count_100000_' in x)
     or ('ngg_pam_100000_' in x or 'count_100000_' in x))]
    sites_feats.extend(['sites_count_ngg_pam_genome_-10_d0','sites_count_ngg_pam_genome_-10_d2','sites_count_ngg_pam_genome_-10_d5'])
    feats_to_keep.extend(sites_feats)
    df.loc[:,'k5_Q1_rep'] = df[[x for x in df.columns.to_list() if 'spacers' in x
    and 'Q1' in x and 'k5' in x and 'GCvsAT' not in x]].mean(axis=1)
    df.loc[:,'k5_Q4_rep'] = df[[x for x in df.columns.to_list() if 'spacers' in x
    and 'Q4' in x and 'k5' in x and 'GCvsAT' not in x]].mean(axis=1)
    feats_to_keep.extend(['k5_Q1_rep','k5_Q4_rep'])
    feats_to_keep.extend(['SPROUT_eff','DeepCRISPR_eff','CRISPRedict_eff','uCRISPR_eff'])
    feats_to_keep.extend(['editing_efficiency','g_rna_info','target_seq','up_seq','down_seq'])
    return df[feats_to_keep]

In [ ]:
df = make_rep_df(df)